<a href="https://colab.research.google.com/github/sulthanalihsan/data-science-2026/blob/main/Pertemuan6_Muhamad_Sulthan_Al_Ihsan_250401020154.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 6 — Persiapan Data
**Nama  :** Muhammad Sulthan Al Ihsan

**NIM   :** 250401020154

**Mata Kuliah:** Data Science —  S1 PJJ Informatika

Import Library

In [22]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
import pandas as pd

# **3.1 Label Encoding**
Mengubah kategori menjadi angka unik (0, 1, 2, dst.) berdasarkan urutan alfabet. Metode ini sederhana namun berisiko menciptakan asumsi urutan yang tidak ada pada data nominal.

In [23]:
df = pd.DataFrame({
'Gender': ['male', 'female', 'female', 'male', 'female'],
'Survived': [0, 1, 1, 0, 1]
})
# ── Label Encoding ────────────────────────────────────────────────
le = LabelEncoder()
# fit_transform: belajar pemetaan + langsung transformasi
df['Gender_enc'] = le.fit_transform(df['Gender'])
print(df)
# Hasil: female → 0, male → 1
# Lihat pemetaan kelas
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print('Mapping:', mapping) # {'female': 0, 'male': 1}
# Decode balik ke label asli
decoded = le.inverse_transform([0, 1, 0])
print(decoded) # ['female' 'male' 'female']

   Gender  Survived  Gender_enc
0    male         0           1
1  female         1           0
2  female         1           0
3    male         0           1
4  female         1           0
Mapping: {'female': np.int64(0), 'male': np.int64(1)}
['female' 'male' 'female']


# **3.2 One-Hot Encoding (OHE)**
Membuat kolom biner terpisah untuk setiap kategori (bernilai 1 jika cocok, 0 jika tidak). Metode ini aman untuk data nominal karena tidak memberikan bobot atau urutan antar kategori.

**3.2.2 Implementasi Python**

In [24]:
df = pd.DataFrame({'City': ['Jakarta','Surabaya','Bandung','Jakarta','Surabaya']})
# ── Cara 1: Pandas get_dummies (paling mudah) ─────────────────────
df_ohe = pd.get_dummies(df,
columns=['City'],
drop_first=False, # True untuk hindari dummy variable trap
dtype=int) # hasilkan 0/1 bukan True/False
print(df_ohe)
# ── Cara 2: sklearn OneHotEncoder (untuk pipeline ML) ─────────────
enc = OneHotEncoder(sparse_output=False, drop='first')
X_enc = enc.fit_transform(df[['City']])
print('Nama fitur:', enc.get_feature_names_out())
# ── Cara 3: Dengan drop_first=True (menghindari Dummy Variable Trap)
df_ohe_drop = pd.get_dummies(df, columns=['City'],
drop_first=True, dtype=int)
# Hasilnya: City_Surabaya, City_Bandung (City_Jakarta dihapus)
# City_Jakarta dapat disimpulkan: jika dua kolom lain = 0, maka Jakarta

   City_Bandung  City_Jakarta  City_Surabaya
0             0             1              0
1             0             0              1
2             1             0              0
3             0             1              0
4             0             0              1
Nama fitur: ['City_Jakarta' 'City_Surabaya']


# **3.3 Ordinal Encoding**
Ordinal Encoding digunakan untuk data kategorikal yang memiliki urutan alami (misalnya: Low < Medium < High). Berbeda dengan Label Encoding yang mengurutkan berdasarkan abjad, metode ini memungkinkan kita menetapkan angka secara manual sesuai makna atau hierarki data yang sebenarnya.

In [25]:
df = pd.DataFrame({
'Pendidikan': ['SMA', 'S1', 'SD', 'D3', 'S2', 'SMP'],
'Gaji_juta': [5, 12, 3, 8, 18, 4]
})
# Definisikan urutan kategori secara eksplisit
edu_order = [['SD', 'SMP', 'SMA', 'D3', 'S1', 'S2']]
enc = OrdinalEncoder(
categories=edu_order,
handle_unknown='use_encoded_value',
unknown_value=-1) # kategori baru → -1
df['Pendidikan_enc'] = enc.fit_transform(df[['Pendidikan']])
print(df.sort_values('Pendidikan_enc'))
# SD=0, SMP=1, SMA=2, D3=3, S1=4, S2=5

  Pendidikan  Gaji_juta  Pendidikan_enc
2         SD          3             0.0
5        SMP          4             1.0
0        SMA          5             2.0
3         D3          8             3.0
1         S1         12             4.0
4         S2         18             5.0


# **4. Scaling & Normalisasi Fitur**
Feature scaling adalah proses menyamakan skala atau rentang nilai antar fitur numerik. Hal ini penting karena banyak algoritma machine learning sangat sensitif terhadap perbedaan besaran data; jika tidak diseragamkan, fitur dengan nilai besar akan mendominasi perhitungan dan menurunkan akurasi model.

**4.1 Mengapa Scaling Diperlukan?**

Tanpa scaling, fitur dengan rentang nilai luas (misalnya Pendapatan 0–50 juta) akan menenggelamkan pengaruh fitur bernilai kecil (misalnya Usia 0–100). Akibatnya, algoritma berbasis jarak seperti KNN menjadi bias karena menganggap selisih pendapatan sebagai satu-satunya faktor penentu, sementara perbedaan usia diabaikan.

**4.2 MinMaxScaler: Normalisasi ke [0, 1]**

MinMaxScaler mentransformasi setiap nilai ke dalam rentang [0, 1]

In [26]:
df = pd.DataFrame({
'Usia': [25, 45, 32, 55, 28],
'Pendapatan': [5, 20, 8, 35, 12] # dalam juta rupiah
})
scaler = MinMaxScaler(feature_range=(0, 1)) # default
# fit_transform: belajar min/max + transformasi data
X_scaled = scaler.fit_transform(df[['Usia', 'Pendapatan']])
print('Min per fitur :', scaler.data_min_) # [25 5]
print('Max per fitur :', scaler.data_max_) # [55 35]
print()
print(pd.DataFrame(X_scaled,
columns=['Usia_sc', 'Pendapatan_sc']).round(3))
# PENTING: gunakan .transform() saja pada data baru (test set!)
# X_test_scaled = scaler.transform(X_test)
# Balik ke skala asli
X_original = scaler.inverse_transform(X_scaled)

Min per fitur : [25.  5.]
Max per fitur : [55. 35.]

   Usia_sc  Pendapatan_sc
0    0.000          0.000
1    0.667          0.500
2    0.233          0.100
3    1.000          1.000
4    0.100          0.233


**4.3 StandardScaler: Z-score Standardization**

StandardScaler mengubah distribusi setiap fitur agar memiliki mean = 0 dan standar deviasi
= 1

In [27]:
df = pd.DataFrame({
'Usia': [25, 45, 32, 55, 28],
'Pendapatan': [5, 20, 8, 35, 12]
})
scaler = StandardScaler()
X = df[['Usia', 'Pendapatan']]
X_scaled = scaler.fit_transform(X)
print('Mean per fitur:', scaler.mean_) # rata-rata setiap kolom
print('Scale per fitur:', scaler.scale_) # std dev setiap kolom
print()
print(pd.DataFrame(X_scaled,
columns=['Usia_z', 'Pend_z']).round(3))
# Contoh output:
# Usia_z Pend_z
# 0 -1.110 -0.784 (di bawah rata-rata)
# 3 1.533 1.568 (di atas rata-rata)
# Selalu .transform() saja pada test set
# X_test_z = scaler.transform(X_test)

Mean per fitur: [37. 16.]
Scale per fitur: [11.296017   10.75174404]

   Usia_z  Pend_z
0  -1.062  -1.023
1   0.708   0.372
2  -0.443  -0.744
3   1.593   1.767
4  -0.797  -0.372


4.4 **RobustScaler**

RobustScaler menggunakan median dan Interquartile Range (IQR) sebagai pengganti mean
dan standar deviasi, sehingga tidak terpengaruh oleh outlier ekstrem

In [29]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(df[['Usia', 'Pendapatan']])
# Gunakan saat dataset mengandung outlier yang tidak bisa dihapus